# 07 — GSV Pipeline: Segmentation & Profile Extraction

Walks through the GSV image processing pipeline:

1. Load a GSV panorama crop
2. Run U-Net sky segmentation
3. Compare U-Net mask against hand-drawn annotation
4. Extract the 1-D elevation-angle skyline profile
5. Validate profile quality

Data: `data/street_view/{images, masks, gsv_crops}/`
Model: `data/sky_segmentation_unet_model.pth`

In [1]:
import sys
from pyprojroot import here
sys.path.insert(0, str(here()))

### Setup — paths and config

In [2]:
import os
import json
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from PIL import Image

from src.segmentation import load_segmentation_model, segment_image
from src.query_profile import extract_elevation_profile, is_profile_applicable, mask_to_boundary

ROOT = here()
IMAGES_DIR = ROOT / "data" / "street_view" / "images"
MASKS_DIR  = ROOT / "data" / "street_view" / "masks"
CROPS_DIR  = ROOT / "data" / "street_view" / "gsv_crops"
ANNOT_FILE = ROOT / "data" / "street_view" / "annotations.json"
GT_FILE    = ROOT / "data" / "street_view" / "ground_truth.json"
MODEL_PATH = ROOT / "data" / "sky_segmentation_unet_model.pth"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_segmentation_model(MODEL_PATH, device)
print(f"Running on: {device}")

Running on: cpu


### Load ground truth and annotations

In [4]:
gt_data = json.loads(GT_FILE.read_text())
annot_data = json.loads(ANNOT_FILE.read_text())
annotations = annot_data.get("annotations", {})

# Find panoramas that have both image and mask
pairs = []
for f in sorted(os.listdir(IMAGES_DIR)):
    if f.endswith((".png", ".jpg")):
        sid = os.path.splitext(f)[0]
        mask_path = MASKS_DIR / f"{sid}.png"
        if mask_path.exists():
            pairs.append(sid)

print(f"{len(pairs)} image+mask pairs found")

1808 image+mask pairs found


### Pick a sample and run the full pipeline

In [5]:
sid = pairs[0]
print(f"Sample: {sid}")

img_path = IMAGES_DIR / f"{sid}.png"
crop_path = CROPS_DIR / f"{sid}.png"

# Load image
img = np.array(Image.open(img_path).convert("RGB"))
print(f"Image shape: {img.shape}")

Sample: --WyciZkeyJi1pLXhEO8BQ
Image shape: (720, 1080, 3)


In [6]:
import tempfile
with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp:
    seg_result = segment_image(model, str(img_path), tmp.name, device)
    mask = np.array(Image.open(tmp.name).convert('L'))
    os.unlink(tmp.name)
print(f"Mask shape: {mask.shape}, unique values: {np.unique(mask)}")

TypeError: segment_image() missing 1 required positional argument: 'device'

### Compare U-Net mask with existing saved mask

In [ ]:
saved_mask = np.array(Image.open(MASKS_DIR / f"{sid}.png").convert("L"))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(img)
axes[0].set_title("GSV Image")
axes[1].imshow(saved_mask, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Saved Mask")
axes[2].imshow(mask, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Fresh U-Net Output")
diff = np.abs(saved_mask.astype(int) - mask.astype(int))
axes[3].imshow(diff, cmap="hot", vmin=0, vmax=255)
axes[3].set_title(f"Pixel Diff (mean={diff.mean():.1f})")
for ax in axes:
    ax.axis("off")
plt.suptitle(sid, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### Skyline boundary extraction — annotation vs U-Net

In [ ]:
bnd_unet = mask_to_boundary(mask)
valid = bnd_unet >= 0
print(f"Boundary coverage: {valid.sum()}/{len(bnd_unet)} columns ({valid.mean():.1%})")

In [ ]:
# Overlay skyline on image
overlay = img.copy()
for c in range(overlay.shape[1]):
    y = int(bnd_unet[c]) if valid[c] else -1
    if 0 <= y < overlay.shape[0]:
        overlay[max(0,y-1):y+2, c] = [0, 255, 255]  # cyan

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].imshow(overlay)
axes[0].set_title(f"U-Net Skyline (cyan) — {sid}")
axes[0].axis("off")

# Annotation overlay
ann_key = f"{sid}_h0_p0_n1"
ann_pts = annotations.get(ann_key, [])
if ann_pts:
    overlay2 = img.copy()
    # Draw annotation
    for x, y in ann_pts:
        xi, yi = int(x), int(y)
        if 0 <= yi < overlay2.shape[0] and 0 <= xi < overlay2.shape[1]:
            overlay2[max(0,yi-1):yi+2, xi] = [255, 50, 50]  # red
    # Draw unet
    for c in range(overlay2.shape[1]):
        y = int(bnd_unet[c]) if valid[c] else -1
        if 0 <= y < overlay2.shape[0]:
            overlay2[max(0,y-1):y+2, c] = [0, 255, 255]  # cyan
    axes[1].imshow(overlay2)
    axes[1].set_title("Annotation (red) vs U-Net (cyan)")
else:
    axes[1].imshow(overlay)
    axes[1].set_title("No annotation available")
axes[1].axis("off")

plt.tight_layout()
plt.show()

### Profile extraction from the mask

In [ ]:
# Use the saved mask for profile extraction (consistent with pipeline)
result = extract_elevation_profile(
    saved_mask,
    fov_y_deg=65.0,
    bin_deg=0.5,
)
profile = result['profile']
applicable = result['ok']
reason = result['status'] + ': ' + result['reason']
print(f"Applicable: {applicable} — {reason}")
print(f"Profile shape: {profile.shape}")
print(f"Std: {np.std(profile):.2f}°, Max: {np.max(profile):.1f}°, Min: {np.min(profile):.1f}°")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [2, 1.5, 1.5]})

# Image with skyline
axes[0].imshow(img)
for c in range(img.shape[1]):
    y = int(bnd_unet[c]) if valid[c] else -1
    if 0 <= y < img.shape[0]:
        axes[0].plot(c, y, ".", color="cyan", markersize=1)
axes[0].set_title(f"Skyline Boundary — {sid}")
axes[0].axis("off")

# Raw profile
axes[1].fill_between(range(len(profile)), profile, alpha=0.3, color="steelblue")
axes[1].plot(profile, color="steelblue", linewidth=1)
axes[1].set_ylabel("Elevation (°)")
axes[1].set_title("Elevation-Angle Profile (720 bins × 0.5°)")
axes[1].grid(True, alpha=0.3)

# Z-scored
if profile.std() > 1e-8:
    zscored = (profile - profile.mean()) / profile.std()
else:
    zscored = profile * 0
axes[2].fill_between(range(len(zscored)), zscored, alpha=0.3, color="coral")
axes[2].plot(zscored, color="coral", linewidth=1)
axes[2].set_ylabel("Z-score")
axes[2].set_xlabel("Azimuth bin (0 = North)")
axes[2].set_title("Z-Scored Profile (used for matching)")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Batch profile quality check

In [ ]:
stats = {"applicable": 0, "too_flat": 0, "no_relief": 0, "other": 0}
n_bins_all = []
std_all = []

for sid in pairs[:100]:
    mask_path = MASKS_DIR / f"{sid}.png"
    if not mask_path.exists():
        continue
    m = np.array(Image.open(mask_path).convert("L"))
    _pr = extract_elevation_profile(m)
    prof, ok, reason = _pr['profile'], _pr['ok'], _pr['reason']
    if ok:
        stats["applicable"] += 1
        std_all.append(np.std(prof))
    elif "flat" in reason.lower():
        stats["too_flat"] += 1
    elif "relief" in reason.lower():
        stats["no_relief"] += 1
    else:
        stats["other"] += 1

print(f"Profile quality (first {min(100, len(pairs))} panos):")
for k, v in stats.items():
    print(f"  {k}: {v}")

if std_all:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(std_all, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
    ax.axvline(x=1.5, color="red", linestyle="--", label="min_std threshold (1.5°)")
    ax.set_xlabel("Profile Std (°)")
    ax.set_ylabel("Count")
    ax.set_title("Profile Standard Deviation Distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()